# 09 · ¿Cambia el resultado de las reglas según el entorno?

Estudiamos por separado momentum y reversión. ¿Qué variables conocidas al decidir se asocian con el retorno medio del mes siguiente o con el tamaño de sus movimientos? Este cribado describe el desarrollo de 1927–1989 y sirve para plantear qué regímenes conviene contrastar después.

Con los resultados de **01 → 02 → 07 → 08** guardados, selecciona `.venv` como kernel y ejecuta este notebook. Usa los datos locales.

El [protocolo de cribado](../docs/hipotesis_y_cribado.md#10-protocolo-para-la-guía-del-28-de-noviembre) fija las hipótesis, los criterios y el papel de estos resultados en la validación posterior.

## 1. Qué variables examinamos

Preparamos las diez características para trabajar con los mismos meses. El retorno del último mes y el momentum 12–1 son las dos entradas de referencia de M1. Las otras **ocho** se examinan como posibles descripciones del entorno.

Para la **rentabilidad media de momentum** esperamos una relación positiva con el retorno de 24 meses, la distancia a la media anual, la distancia al máximo anual y la amplitud industrial. Para la **magnitud de los movimientos de ambas reglas** esperamos una relación positiva con la volatilidad de 12 meses y el diferencial Baa–Aaa. Las demás relaciones se presentan como exploratorias.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.features import feature_dictionary
from tfm.screening import (BH_ALPHA, HAC_LAGS, MIN_SHIFT, REGIME_FEATURE_COLUMNS, REGIME_MIN_HISTORY,
                           RESULT_NAMES, build_screening_monthly, describe_regimes, screen_relationships)

ROOT = project_root()
caracteristicas, _ = read_result(ROOT, "monthly_features")
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
etiquetas, informe_etiquetas = read_result(ROOT, "labelled_signals")
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
if informe_etiquetas["parameters"] != parametros:
    raise ValueError("Utiliza los mismos supuestos centrales en los notebooks 08 y 09.")
nombres = feature_dictionary().set_index("feature").description.to_dict()
reglas = {"momentum": "Momentum", "reversal": "Reversión"}
display(pd.Series({"Historia anterior mínima (meses)": REGIME_MIN_HISTORY,
                   "Retardos Newey–West (meses)": HAC_LAGS,
                   "BH: p ajustado <": BH_ALPHA,
                   "Desplazamiento circular mínimo (meses)": MIN_SHIFT}, name="Protocolo"))
display(pd.Series(parametros, name="Supuestos contables"))

## 2. Qué resultado medimos y cómo preparamos las variables

Simulamos cada regla continua con costes y dividendos. Restamos el retorno del efectivo (RF) del mismo mes para obtener el **exceso mensual**. El exceso firmado sirve para estudiar la rentabilidad media. Su valor absoluto sirve para estudiar el tamaño de los movimientos. Por ejemplo, −4 % y +4 % tienen la misma magnitud.

El calendario incluye los meses en efectivo. Las asociaciones pueden reflejar tanto el resultado de operar como la frecuencia con la que la regla toma posición. Por eso mostramos también cuántos meses con señal hay en cada tercil.

Cada característica requiere **120 observaciones mensuales anteriores válidas**. Con ese pasado calculamos los percentiles 1 y 99 y la media y desviación típica de la historia recortada. Recortamos el valor actual con esos mismos límites y lo expresamos en esa escala. Los retornos de las estrategias se conservan íntegros. La muestra común empieza cuando las diez variables están listas.

In [ ]:
mensual, informe = build_screening_monthly(caracteristicas, componentes, senales, parametros)
comun = mensual.loc[mensual.screening_ready].copy()
display(pd.DataFrame([{
    "Meses por regla antes del calentamiento": informe["calendar_months"],
    "Meses comunes por regla": informe["common_months"],
    "Primer objetivo común": informe["first_target"],
    "Último objetivo común": informe["last_target"],
    "Meses excluidos por regla": len(informe["excluded_targets"]),
}]))
display(pd.DataFrame(informe["excluded_targets"]).head(5))
print("Cada fila conserva la decisión y las fechas de fuente. La lista completa de exclusiones se guardará en el informe.")

### Un mismo mes, dos preguntas distintas

Una operación aislada abre desde efectivo y cierra al terminar el mes. La regla continua puede conservar la posición que ya tenía y pagar menos negociación. Comparamos ambas cuentas para entender la diferencia, el cribado usa exclusivamente el retorno de la **regla continua**. El resultado observado del mes objetivo nunca se incorpora como predictor.

In [ ]:
mes_objetivo = "1975-05"
if mes_objetivo not in set(mensual.target_period):
    raise ValueError("Escoge un objetivo entre 1927-01 y 1989-12.")
ejemplo = mensual.loc[mensual.target_period.eq(mes_objetivo),
                      ["strategy", "decision_period", "target_period", "position", "anomaly_return"]].merge(
    etiquetas[["strategy", "target_period", "excess_return", "label"]],
    on=["strategy", "target_period"], validate="one_to_one",
)
display(ejemplo.rename(columns={"strategy": "Regla", "decision_period": "Decisión",
    "target_period": "Objetivo", "position": "Posición",
    "anomaly_return": "Exceso de la regla continua", "excess_return": "Exceso de operación aislada",
    "label": "Etiqueta del 08"}).set_index("Regla"))

## 3. Contrastes sobre la media y la magnitud

Ajustamos una regresión por variable para cada una de las dos respuestas. Una pendiente positiva en la **media** indica que valores más altos de la variable se asocian con rendimientos mejores. Una pendiente positiva en la **magnitud** indica que se asocian con movimientos mayores, favorables o desfavorables.

Calculamos el error estándar con Newey–West, seis retardos mensuales y pesos Bartlett. **Benjamini–Hochberg (BH) con p ajustado < 0,10** se aplica de forma conjunta a los 32 contrastes (ocho variables × dos reglas × dos respuestas). Las pruebas no estimables mantienen su fila y se incorporan al ajuste con p = 1. El umbral BH es nominal y depende de sus supuestos estadísticos.

La columna **Supera el cribado** requiere superar BH y obtener el signo previsto. Las relaciones significativas sin signo previo se muestran como exploratorias. Los resultados de este notebook describen el desarrollo, y la selección de entradas para los modelos se hará dentro de su entrenamiento.

In [ ]:
contrastes, sombras, informe_sombras = screen_relationships(mensual)
respuestas = {"return": "Rentabilidad media", "magnitude": "Magnitud"}
estados = {"not_estimable": "No estimable", "no_bh_evidence": "Sin respaldo tras BH",
           "bh_evidence": "Supera BH"}
direcciones = {"not_estimable": "No estimable", "exploratory": "Exploratoria",
               "supports_expected_sign": "Coincide", "does_not_support_expected_sign": "Contrario"}
tabla = contrastes[["strategy", "response", "feature", "expected_sign", "slope", "t_hac",
                    "p_value", "p_bh", "evidence_status", "direction_status",
                    "passes_directional_screen", "exceeds_shadows"]].copy()
tabla["strategy"] = tabla.strategy.map(reglas)
tabla["response"] = tabla.response.map(respuestas)
tabla["feature"] = tabla.feature.map(nombres)
tabla["expected_sign"] = tabla.expected_sign.replace({"unspecified": "Sin signo"})
tabla["evidence_status"] = tabla.evidence_status.map(estados)
tabla["direction_status"] = tabla.direction_status.map(direcciones)
tabla["passes_directional_screen"] = tabla.passes_directional_screen.map({True: "Sí", False: "No"})
tabla["exceeds_shadows"] = tabla.exceeds_shadows.map({True: "Sí", False: "No"}).fillna("No disponible")
with pd.option_context("display.max_rows", 40, "display.max_columns", 14):
    display(tabla.rename(columns={"strategy": "Regla", "response": "Resultado",
        "feature": "Característica", "expected_sign": "Signo esperado", "slope": "Pendiente",
        "t_hac": "t HAC", "p_value": "p sin ajustar", "p_bh": "p BH",
        "evidence_status": "Evidencia", "direction_status": "Dirección",
        "passes_directional_screen": "Supera el cribado", "exceeds_shadows": "Supera sombras"
    }).set_index(["Regla", "Resultado", "Característica"]))

## 4. Terciles y estabilidad por periodos

Los terciles agrupan meses con valores bajos, medios y altos de una variable, usando cortes calculados con su pasado. Mostramos cuántos meses tienen señal, el exceso medio, la magnitud media, el peor mes y el Sharpe. El Sharpe de un tercil resume meses que pueden estar separados.

Las pendientes de los cuatro tramos fijos ayudan a ver si cada relación aparece a lo largo del tiempo. Los gráficos empiezan cuando hay datos utilizables y muestran las dos respuestas en escalas separadas.

In [ ]:
terciles, estabilidad = describe_regimes(mensual)
caracteristica = "price_volatility_12m"
if caracteristica not in REGIME_FEATURE_COLUMNS:
    raise ValueError("Escoge una de las ocho características de régimen.")
detalle_terciles = terciles.loc[terciles.feature.eq(caracteristica)]
detalle_estabilidad = estabilidad.loc[estabilidad.feature.eq(caracteristica)]
display(detalle_terciles.drop(columns="feature").rename(columns={
    "strategy": "Regla", "tercile": "Tercil", "months": "Meses", "signal_months": "Meses con señal",
    "signal_fraction": "Fracción con señal", "mean_excess_return": "Exceso medio",
    "mean_absolute_excess_return": "Magnitud media", "worst_month_excess_return": "Peor mes",
    "annualized_conditional_sharpe": "Sharpe condicional anualizado"}))

fig, axes = plt.subplots(2, 2, figsize=(11, 6), layout="constrained")
medias = {"return": "mean_excess_return", "magnitude": "mean_absolute_excess_return"}
for (respuesta, nombre_respuesta), (ax_terciles, ax_estabilidad) in zip(respuestas.items(), axes, strict=True):
    for regla, nombre in reglas.items():
        datos = detalle_terciles.loc[detalle_terciles.strategy.eq(regla)]
        ax_terciles.plot(datos.tercile, datos[medias[respuesta]] * 100, marker="o", label=nombre)
        tramos = detalle_estabilidad.loc[
            detalle_estabilidad.strategy.eq(regla) & detalle_estabilidad.response.eq(respuesta)
        ].dropna(subset=["first_observed_target"])
        anos = tramos.first_observed_target.str[:4] + "–" + tramos.last_observed_target.str[2:4]
        ax_estabilidad.plot(anos, tramos.slope * 100, marker="o", label=nombre)
    ax_terciles.set(title=f"{nombre_respuesta} por tercil", xlabel="Tercil", ylabel="%", xticks=[1, 2, 3])
    ax_estabilidad.set(title=f"Pendiente · {nombre_respuesta.lower()}",
                       xlabel="Años observados", ylabel="Puntos porcentuales por unidad de z")
for ax in axes.flat:
    ax.axhline(0, color="gray", linewidth=.8)
    ax.grid(alpha=.2)
axes[0, 0].legend()
fig.suptitle(nombres[caracteristica])
plt.show()

## 5. Sombras circulares

Desplazamos juntas las ocho series estandarizadas al menos 36 meses, manteniendo fijos los retornos de las dos reglas. Esto da una referencia de cuánto podrían destacar asociaciones cuando las fechas están desalineadas.

Cada desplazamiento aporta el mayor `abs(t HAC)` de los 32 contrastes. Usamos el percentil 95 de esos máximos. Si algún desplazamiento queda incompleto, se conserva su diagnóstico y el percentil queda sin calcular. Las sombras contextualizan el cribado. BH sigue siendo el criterio de significación.

In [ ]:
display(pd.Series(informe_sombras, name="Sombras"))
resumen = contrastes.groupby(["strategy", "response", "evidence_status"]).size().rename("Relaciones").reset_index()
resumen["strategy"] = resumen.strategy.map(reglas)
resumen["response"] = resumen.response.map(respuestas)
resumen["evidence_status"] = resumen.evidence_status.map(estados)
display(resumen.rename(columns={"strategy": "Regla", "response": "Resultado", "evidence_status": "Evidencia estadística"}))
validas = sombras.max_abs_t_hac.dropna()
if informe_sombras["shadow_comparison_available"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(validas, bins=25, alpha=.7, label="Máximo de cada familia desplazada")
    ax.axvline(informe_sombras["shadow_threshold_95"], color="red", linestyle="--", label="Percentil 95")
    ax.set(xlabel="Máximo abs(t HAC) entre los 32 contrastes", ylabel="Desplazamientos", title="Referencia de sombras circulares")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Comparación con sombras no disponible: alguna familia está incompleta. Consulta el número de relaciones estimables por sombra y los motivos de los contrastes.")

## 6. Guardar el diagnóstico

Guardamos cinco tablas `screening_120_*` y sus informes JSON en `data/processed/`. Incluyen el calendario preparado, los contrastes, los terciles, los subperiodos y las sombras, con sus exclusiones y parámetros.

La última tabla de contrastes indica qué relaciones superan el cribado descriptivo. La validación de un filtro requiere repetir la selección dentro de cada entrenamiento y comprobar su resultado en la parte de prueba. El paso siguiente es preparar ese calendario en el notebook 10.

In [ ]:
informe_final = {**informe, **informe_sombras,
    "scope": "descriptive_development_screening_only",
    "protocol": "docs/hipotesis_y_cribado.md#10-protocolo-para-la-guía-del-28-de-noviembre",
    "references": {"newey_west": "10.2307/1913610",
                   "benjamini_hochberg": "10.1111/j.2517-6161.1995.tb02031.x",
                   "jansen_revision": "91c903e71c1d0f1b9015f18551fadbb60bcad132"},
    "warnings": [
        "Descriptive 1927–1989 screening, no out-of-sample skill or final feature selection.",
        "Monthly Shiller means and reconstructed dividends, no executable-price interpretation.",
        "Response from continuous unfiltered rules, not isolated labels, no clipping of returns.",
        "HAC and BH rely on assumptions; BH independence or positive-dependence conditions are not certified here.",
        "Directional screening requires adjusted p < 0.10 and the prespecified sign.",
        "Circular shadows are a separate plausibility check, not an exact randomization test or a BH requirement.",
        "Tercile Sharpes describe separated months, they are not portfolio backtests.",
        "Selection based on outcomes must be repeated within training partitions, no post-1989 evaluation.",
    ],
}
salidas = zip(RESULT_NAMES, (mensual, contrastes, terciles, estabilidad, sombras), strict=True)
guardados = []
for nombre, datos in salidas:
    salida = save_result(ROOT, nombre, datos, informe_final)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Diagnóstico descriptivo guardado en data/processed/. Cada tabla tiene su informe JSON.")